Copyright 2025 DeepMind Technologies Limited.

In [ ]:
#@title SPDX-License-Identifier: Apache-2.0
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# RAG Tutorial with OneTwo

This tutorial demonstrates how to build a **Retrieval-Augmented Generation (RAG)** pipeline using the [OneTwo](https://github.com/google-deepmind/onetwo) library.

We will:
1. Load the **HotpotQA** multi-hop question answering dataset using `HotpotQADatasetLoader`.
2. Inspect the loaded examples and documents.
3. ...

# Setup

## Imports

In [ ]:
import pprint

from onetwo import ot
from onetwo.evaluation import datasets

# Loading HotpotQA

[HotpotQA](https://hotpotqa.github.io/) is a multi-hop question answering dataset where each question requires reasoning over multiple Wikipedia paragraphs to arrive at the answer.

OneTwo provides `HotpotQADatasetLoader`, a convenient wrapper that loads HotpotQA via TFDS and returns:

- **Examples**: Dictionaries with `question` and `answer` keys, compatible with `ot.evaluate`.
- **Documents**: `Document` objects (with `doc_id`, `title`, and `content` fields) representing the Wikipedia paragraphs associated with each example, suitable for building a retrieval corpus.

Note that Documents should not be expected to be in a particular order matching the order of the examples.

## Create the Loader

We create a `HotpotQADatasetLoader` with `max_number_of_examples=5` to load only the first 5 examples (and their associated documents) for a quick preview.

In [ ]:
loader = datasets.HotpotQADatasetLoader(
    split='validation',
    max_number_of_examples=5,
)

## Load Examples

The `load_examples()` method returns evaluation-compatible example dictionaries. Each example contains:
- `question`: The question string.
- `answer`: The gold answer string.
- `metadata`: Additional HotpotQA-specific fields (e.g., `id`, `type`, `level`).

In [ ]:
examples = ot.run(loader.load_examples())
print(f'Loaded {len(examples)} examples.\n')

for i, example in enumerate(examples):
  print(f'--- Example {i + 1} ---')
  print(f'  Question: {example["question"]}')
  print(f'  Answer:   {example["answer"]}')
  print(f'  Metadata: {example["metadata"]}')
  print()

Loaded 5 examples.

--- Example 1 ---
  Question: William Davis Ticknor, Sr. served on the board of a chemical and biotechnology company created in what year?
  Answer:   1919
  Metadata: {'id': '5ae1faf15542997f29b3c1e3', 'type': 'bridge', 'level': 'hard', 'supporting_facts': {'sent_id': array([0, 0], dtype=int32), 'title': array([b'William Davis Ticknor, Sr.', b'Commercial Solvents Corporation'],
      dtype=object)}}

--- Example 2 ---
  Question: The 2009 Fort Hood mass shooting was committed by who?
  Answer:   Nidal Hasan
  Metadata: {'id': '5ab9339f554299131ca422bb', 'type': 'bridge', 'level': 'hard', 'supporting_facts': {'sent_id': array([0, 1], dtype=int32), 'title': array([b'Nidal Hasan', b'2009 Fort Hood shooting'], dtype=object)}}

--- Example 3 ---
  Question: A Talk is an EP by which South Korean singer and dancer?
  Answer:   Hyuna
  Metadata: {'id': '5ae7376f5542992ae0d163d2', 'type': 'bridge', 'level': 'hard', 'supporting_facts': {'sent_id': array([0, 0], dtype=int32),

## Load Documents

The `load_documents()` method returns `Document` objects extracted from the HotpotQA context paragraphs. Documents are **deduplicated** by `doc_id` (since the same Wikipedia paragraph can appear across multiple examples).

Each `Document` has:
- `doc_id`: Unique identifier (the Wikipedia paragraph title).
- `title`: The paragraph title.
- `content`: The concatenated text of the paragraph sentences.

In [5]:
documents = ot.run(loader.load_documents())
print(f'Loaded {len(documents)} unique documents.\n')

for i, doc in enumerate(documents[:5]):
  print(f'--- Document {i + 1} ---')
  print(f'  Title:    {doc.title}')
  print(f'  Content:  {doc.content[:200]}{"..." if len(doc.content) > 200 else ""}')
  print(f'  Metadata: {doc.metadata}')

Loaded 50 unique documents.

--- Document 1 ---
  Title:    Icos
  Content:  Icos Corporation (trademark ICOS) was an American biotechnology company and the largest biotechnology company in the U.S. state of Washington, before it was sold to Eli Lilly and Company in 2007.  It ...
  Metadata: {}
--- Document 2 ---
  Title:    Commercial Solvents Corporation
  Content:  Commercial Solvents Corporation (CSC) was an American chemical and biotechnology company created in 1919.
  Metadata: {}
--- Document 3 ---
  Title:    Pascal Brandys
  Content:  Pascal Brandys (born 30 November 1958 in Roanne) is a French engineer and entrepreneur.  He is a graduate of the École Polytechnique and received his M.S. in Economic Systems from Stanford University ...
  Metadata: {}
--- Document 4 ---
  Title:    Daiichi Sankyo
  Content:  Daiichi Sankyo Company, Limited (第一三共株式会社 , Daiichi Sankyō Kabushiki-kaisha ) is a global pharmaceutical company and the second largest pharmaceutical company in Japan.  It 